# BaSSSh-seq — Bacterial scRNA-seq analysis
BaSSSh-seq pipeline — complete script
Steps: load → merge → fix cell names → HVG/PCA/BBKNN → UMAP/Leiden → markers → annotation → save
Step 7 (CellTypist training) is NOT included.

**Order:** Import → Preprocessing (run once) → Load → Merge → HVG/PCA/BBKNN → UMAP/Leiden → Marker genes → Annotation → CellTypist

## 1. Imports

In [25]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import celltypist

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)

In [26]:
print("=== STEP 1: Load datasets ===")
adata_bf = sc.read_h5ad("../data/BF_gefilterd.h5ad", backed="r")
adata_p  = sc.read_h5ad("../data/P_gefilterd.h5ad",  backed="r")
print(f"BF: {adata_bf.shape[0]} cells, {adata_bf.shape[1]} genes")
print(f"P:  {adata_p.shape[0]} cells,  {adata_p.shape[1]} genes")
 

=== STEP 1: Load datasets ===
BF: 3668 cells, 2853 genes
P:  4216 cells,  2853 genes


## 2a. BF preprocessing — Run once, then comment out
This cell reads the raw BF count matrix, removes rRNA genes, filters cells on ≥ 7 non-rRNA reads, normalises, and saves the result as BF_gefilterd.h5ad

In [27]:
# ============================================================
# Run once → comment out entirely afterwards
# ============================================================

#adata_bf = sc.read_h5ad("../data/BF_count_mat.h5ad")
#adata_bf.var_names_make_unique()

# # rRNA removal (paper: all rRNA genes removed before analysis)
# # Bacterial rRNA: 5S, 16S, 23S
#adata_bf.var["rrna"] = (
     #adata_bf.var_names.str.lower().str.startswith(("rrs", "rrl", "rrf")) |
     #adata_bf.var_names.str.lower().str.contains("rrna") |
     #adata_bf.var_names.str.lower().str.contains("16s") |
     #adata_bf.var_names.str.lower().str.contains("23s") |
     #adata_bf.var_names.str.lower().str.contains("5s_rrna")
#)
#print(f"rRNA genes found (BF): {adata_bf.var['rrna'].sum()}")
#adata_bf = adata_bf[:, ~adata_bf.var["rrna"]].copy()

# # Count filter: paper ≥ 7 non-rRNA reads for BF → expected ~3680 cells
#adata_bf.obs["total_counts"] = adata_bf.X.sum(axis=1)
#print(f"Cells before filter: {adata_bf.n_obs}")
#adata_bf = adata_bf[adata_bf.obs["total_counts"] >= 7].copy()
#print(f"Cells after filter (≥7): {adata_bf.n_obs}")  # Paper: 3680 cells

# # Normalisation: 10^4 counts, log+1 transform (paper Methods)
#sc.pp.normalize_total(adata_bf, target_sum=1e4)
#sc.pp.log1p(adata_bf)

# # Label for merge
#adata_bf.obs["sample"] = "bf"

# # Save
#adata_bf.write("../data/BF_gefilterd.h5ad", compression="gzip")
#print("BF saved!")

## 2b. Load BF
After filtering on ≥7 non-rRNA reads per cell, we obtained 3,668 biofilm cells compared to 3,680 reported in the paper — a difference of 12 cells (~0.33%), likely due to minor package version differences and considered negligible.

In [28]:
# Load the preprocessed BF data (memory-mapped, low RAM)
adata_bf = sc.read_h5ad("../data/BF_gefilterd.h5ad", backed="r")

## 3a. Planktonic preprocessing — Run once, then comment out
This cell reads the raw Planktonic count matrix, removes rRNA genes, filters cells on ≥ 28 non-rRNA reads, normalises, and saves the result as P_gefilterd.h5ad.

In [29]:
# ============================================================
# Run once → comment out entirely afterwards
# ============================================================

#adata_p = sc.read_h5ad("../data/P_count_mat.h5ad")
#adata_p.var_names_make_unique()

# # Same rRNA removal as BF
#adata_p.var["rrna"] = (
     #adata_p.var_names.str.lower().str.startswith(("rrs", "rrl", "rrf")) |
     #adata_p.var_names.str.lower().str.contains("rrna") |
     #adata_p.var_names.str.lower().str.contains("16s") |
     #adata_p.var_names.str.lower().str.contains("23s") |
    # adata_p.var_names.str.lower().str.contains("5s_rrna")
#)
#print(f"rRNA genes found (P): {adata_p.var['rrna'].sum()}")
#adata_p = adata_p[:, ~adata_p.var["rrna"]].copy()

# # Count filter: paper ≥ 28 non-rRNA reads for Planktonic → expected ~4231 cells
#adata_p.obs["total_counts"] = adata_p.X.sum(axis=1)
#print(f"Cells before filter: {adata_p.n_obs}")
#adata_p = adata_p[adata_p.obs["total_counts"] >= 28].copy()
#print(f"Cells after filter (≥28): {adata_p.n_obs}")  # Paper: 4231 cells

# # Normalisation
#sc.pp.normalize_total(adata_p, target_sum=1e4)
#sc.pp.log1p(adata_p)

# # Label for merge
#adata_p.obs["sample"] = "p"

# # Save
#adata_p.write("../data/P_gefilterd.h5ad", compression="gzip")
#print("Planktonic saved!")

## 3b. Load Planktonic
After filtering on ≥28 non-rRNA reads per cell, we obtained 4,216 planktonic cells compared to 4,231 reported in the paper — a difference of 15 cells (~0.35%), likely due to minor package version differences and considered negligible.

In [30]:
# Load the preprocessed Planktonic data (memory-mapped, low RAM)
adata_p = sc.read_h5ad("../data/P_gefilterd.h5ad", backed="r")

## 4. Merge BF + Planktonic
Paper: cells from both conditions combined for BBKNN-integrated clustering

In [31]:
# Merge the two datasets (join='inner' = keep only shared genes)
adata_bacteria = ad.concat(
    {"bf": adata_bf, "p": adata_p},
    label="sample",
    join="inner"
)
adata_bacteria.obs_names = [
    f"{sample}-{barcode}"
    for barcode, sample in zip(adata_bacteria.obs_names, adata_bacteria.obs['sample'])
]

print(f"Shape combined dataset: {adata_bacteria.shape}")
# Expected: ~7911 cells (3680 BF + 4231 P), number of genes may vary

print(f"\nSamples present: {adata_bacteria.obs['sample'].value_counts().to_dict()}")
adata_bacteria.obs.head()

Shape combined dataset: (7884, 2853)

Samples present: {'p': 4216, 'bf': 3668}


,total_counts,sample
bf-68,14.120000,bf
bf-96,8.770000,bf
bf-104,16.639999,bf
bf-107,8.700000,bf
bf-131,9.680000,bf


In [ ]:
# =============================================================
# STEP 3: Fix cell names (replace bare numbers → bf-X / p-X)
# =============================================================
adata_bacteria = ad.concat(
    {"bf": adata_bf, "p": adata_p},
    label="sample",
    join="inner"
)

adata_bacteria.obs_names = [
    f"{sample}-{barcode}"
    for barcode, sample in zip(adata_bacteria.obs_names, adata_bacteria.obs['sample'])
]

print(adata_bacteria.obs_names[:5].tolist())
#print(adata_bacteria.obs.obs)
adata_bacteria.write("../data/bacteria_combined.h5ad", compression="gzip")

['bf-68', 'bf-96', 'bf-104', 'bf-107', 'bf-131']


AttributeError: 'DataFrame' object has no attribute 'obs'

In [33]:
# =============================================================
# STEP 4: Verify expected dimensions
# =============================================================
print("\n=== STEP 4: Dimensions check before HVG ===")
print(f"Cells: {adata_bacteria.n_obs}  (expected ~7884)")
print(f"Genes: {adata_bacteria.n_vars} (expected 2853)")


=== STEP 4: Dimensions check before HVG ===
Cells: 7884  (expected ~7884)
Genes: 2853 (expected 2853)


In [34]:
# =============================================================
# STEP 5: HVG → Scale → PCA → BBKNN
# =============================================================
print("\n=== STEP 5: HVG + PCA + BBKNN ===")
 
# Store raw counts before filtering (required for marker genes & CellTypist)
adata_bacteria.raw = adata_bacteria
 
sc.pp.highly_variable_genes(
    adata_bacteria,
    min_mean=0.00625,
    min_disp=0.25,
    batch_key='sample'
)
n_hvg = adata_bacteria.var.highly_variable.sum()
print(f"Number of highly variable genes: {n_hvg}  (expected ~1015)")
 
# Filter to HVGs
adata_bacteria = adata_bacteria[:, adata_bacteria.var.highly_variable].copy()
 
# Scale & PCA
sc.pp.scale(adata_bacteria, max_value=10)
sc.tl.pca(adata_bacteria, n_comps=50)
 
# BBKNN batch correction (paper: neighbors_within_batch=9, n_pcs=4)
sc.external.pp.bbknn(
    adata_bacteria,
    batch_key='sample',
    neighbors_within_batch=9,
    n_pcs=4
)
print("HVG + PCA + BBKNN done!")


=== STEP 5: HVG + PCA + BBKNN ===
Number of highly variable genes: 1015  (expected ~1015)
HVG + PCA + BBKNN done!


In [35]:
adata_bacteria.obs_names = adata_bacteria.obs_names.str.replace("-", "_")
adata_bacteria.write("../data/bacteria_combined_annotated.h5ad", compression="gzip")

# Verify
test = sc.read_h5ad("../data/bacteria_combined_annotated.h5ad")
print(f"Shape: {test.shape}")
print(f"Columns: {test.obs.columns.tolist()}")
print(f"Cell names: {list(test.obs_names[:3])}")

Shape: (7884, 1015)
Columns: ['total_counts', 'sample']
Cell names: ['bf_68', 'bf_96', 'bf_104']


In [ ]:
#1. fix the cells names, remove the -1.
#2. takes into account of -1, if I have duplicated cell names (check this)
#3. be sure that the new adata object have the changes, and the same highly variable genes as the celltypist?

In [ ]:
#pre adata object in order to be sure that I have the correct number of cells and genes. I expect 78.. cells and 1015

In [ ]:
#here I have to save it as a .h5ad file, because you work with this. This is not comparable


In [ ]:
#graphical embedding: 
#check correlation of cells in teach neigho

#MDS 
#takes one cell anc check the correlation ofhe whole cells 

In [ ]:

test.obs_names

In [ ]:
#how can I see the columns names of the cell types 
#how did you see that the name of the cells were with - or did I did?
w#how can i see the inside of the MICA file, if I run it in R its not readible 

## 6. UMAP + Leiden clustering
Parameters taken directly from paper Methods: min_dist=0.24, spread=0.21, resolution=0.205

In [ ]:
# UMAP (paper: min_dist=0.24, spread=0.21)
sc.tl.umap(adata_bacteria, min_dist=0.24, spread=0.21)

# Leiden clustering (paper: resolution=0.205 → yields 7 clusters)
sc.tl.leiden(adata_bacteria, resolution=0.205)

n_clusters = adata_bacteria.obs['leiden'].nunique()
print(f"Number of clusters found: {n_clusters}")
print("Expected: 7 — if this differs, adjust resolution slightly (+/- 0.01)")

# Labels for visualisation
label_map = {'bf': 'Biofilm', 'p': 'Planktonic'}
adata_bacteria.obs['Cell identity'] = adata_bacteria.obs['sample'].map(label_map)

# UMAP plot (comparable to Fig. 2B and 2C from paper)
sc.pl.umap(
    adata_bacteria,
    color=['Cell identity', 'leiden'],
    title=['Cell identity (Fig. 2B)', 'Leiden clusters (Fig. 2C)'],
    wspace=0.4,
    frameon=False
)

## 7. Cluster composition (Fig. 2D)
Stacked bar chart: distribution of BF vs Planktonic per cluster

In [ ]:
dist = pd.crosstab(adata_bacteria.obs['leiden'], adata_bacteria.obs['Cell identity'])
dist_norm = dist.div(dist.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
color_map = {'Biofilm': 'steelblue', 'Planktonic': 'tomato'}

# Absolute
dist.plot(kind='bar', ax=axes[0], color=color_map, stacked=True)
axes[0].set_ylabel('Number of cells')
axes[0].set_xlabel('Leiden Cluster')
axes[0].set_title('Absolute cell counts per cluster')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Condition', frameon=False)

# Normalised (as in Fig. 2D paper)
dist_norm.plot(kind='bar', ax=axes[1], color=color_map, stacked=True)
axes[1].set_ylabel('Proportion of cells')
axes[1].set_xlabel('Leiden Cluster')
axes[1].set_title('Proportional distribution per cluster (Fig. 2D)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].axhline(0.5, color='black', linestyle='--', linewidth=0.8)
axes[1].legend(title='Condition', frameon=False, loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()

# Interpretation guide:
# >80% Planktonic → Planktonic subtype
# >80% Biofilm    → Biofilm subtype
# ~50/50          → Transitional (cluster 0 in paper)

## 8. Marker genes per cluster
Paper uses MAST (R); Wilcoxon is used here as the Python equivalent with the same interpretation:
small log2FC values are NOT insignificant (explicitly noted as a caveat in the paper)

In [ ]:
sc.tl.rank_genes_groups(
    adata_bacteria,
    groupby='leiden',
    method='wilcoxon',  # best Python equivalent for MAST
    use_raw=True,       # use log-normalised counts before scaling
    pts=True            # % of cells expressing the gene
)

# Top 10 markers per cluster
sc.pl.rank_genes_groups(adata_bacteria, n_genes=10, sharey=False)

# Top 5 table
marker_table = pd.DataFrame(adata_bacteria.uns['rank_genes_groups']['names']).head(5)
print("Top marker genes per cluster:")
print(marker_table)

# Expected markers from Table 1 of the paper (for comparison)
print("\n=== Expected markers (paper Table 1) ===")
paper_markers = {
    "Cluster 0 (Transitional)"      : ["sasA", "pnpA", "ebh"],
    "Cluster 1 (BF Active)"         : ["citB", "ltaS", "isdH", "ebpS"],
    "Cluster 2 (P Active)"          : ["rpoB", "rpsC", "ispA"],
    "Cluster 3 (BF Virulence)"      : ["clfB", "cspB", "arcA", "fnbB"],
    "Cluster 4 (P Stationary)"      : ["qoxA", "hemY", "rplY"],
    "Cluster 5 (BF Stress)"         : ["gpmA", "polX", "fdaB", "ureA"],
    "Cluster 6 (BF Replication)"    : ["nrdE"],
}
for cluster, genes in paper_markers.items():
    print(f"  {cluster}: {genes}")

print("\nCluster composition (% BF vs P):")
print(dist_norm.round(2))

## 9. Cluster annotation
Adjust the names based on your marker table and composition chart above

In [ ]:
# STEPS:
# 1. Review the marker table above
# 2. Review the stacked bar chart (which clusters are BF vs P dominated?)
# 3. Compare with the paper markers above
# 4. Adjust the names below if needed

cluster_names = {
    '0': 'Transitional',                    # ~50/50 BF/P, sasA, pnpA
    '1': 'Biofilm_Transcriptionally_Active', # citB, ltaS, isdH
    '2': 'Planktonic_Active',               # rpoB, rpsC — most active
    '3': 'Biofilm_Virulence',               # clfB, fnbB, arcA
    '4': 'Planktonic_Stationary',           # qoxA, hemY
    '5': 'Biofilm_Stress_Metabolism',       # gpmA, polX, fdaB
    '6': 'Biofilm_Replication'              # nrdE
}

print(f"Clusters in data:      {sorted(adata_bacteria.obs['leiden'].unique())}")
print(f"Clusters in name dict: {sorted(cluster_names.keys())}")

adata_bacteria.obs['cell_type'] = adata_bacteria.obs['leiden'].map(cluster_names)
print(f"\nUnmapped cells: {adata_bacteria.obs['cell_type'].isna().sum()}")

sc.pl.umap(
    adata_bacteria,
    color='cell_type',
    legend_loc='on data',
    title='Cell type annotation'
)

## 10. CellTypist training

In [ ]:
print(f"Cells without label: {adata_bacteria.obs['cell_type'].isna().sum()}")
assert adata_bacteria.obs['cell_type'].isna().sum() == 0, "Fix missing labels first!"

model = celltypist.train(
    adata_bacteria,
    labels='cell_type',
    feature_selection=True,
    top_genes=300
)

model.write('Staph_Aureus_BF_P_Model.pkl')
print("Model trained and saved!")
print(f"\nTop features: {model.features}")